<a href="https://colab.research.google.com/github/NamithaDaney/DataLens-AI/blob/main/DataLens_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas openpyxl

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI")

INPUT_DIR = BASE_DIR / "input"
REPORT_DIR = BASE_DIR / "reports"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Reports folder:", REPORT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/input
Reports folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports


In [ ]:
from pathlib import Path
import pandas as pd

def load_dataset(file_path):
    """
    Load a CSV or Excel file into a Pandas DataFrame.
    Detect possible index columns without automatically deleting them.
    Supports .csv, .xlsx and .xls files.
    """

    file_path = Path(file_path)
    extension = file_path.suffix.lower()

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if extension == ".csv":
        df = pd.read_csv(file_path,index_col=0)

    elif extension in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Please upload a CSV or Excel file."
        )

    if df.empty:
        raise ValueError("The uploaded file contains no data.")

    # Detect possible index columns
    unnamed_columns = [
        col for col in df.columns
        if str(col).strip().lower().startswith("unnamed:")
    ]

    if unnamed_columns:
        print("Possible index columns detected:", unnamed_columns)
        print("Review these columns before deciding whether to remove them.")

    return df


In [ ]:
def review_unnamed_columns(df):
    """
    Detect possible index columns and recommend an action.
    Does not delete any columns automatically.
    """

    review_results = []

    for col in df.columns:
        column_name = str(col).strip().lower()

        if not column_name.startswith("unnamed:"):
            continue

        values = df[col]

        # Ignore missing values while checking the pattern
        non_missing = values.dropna()

        # Check whether values match 0, 1, 2, 3, ...
        numeric_values = pd.to_numeric(
            non_missing, errors="coerce"
        )

        looks_like_index = (
            len(non_missing) == len(df)
            and len(non_missing) > 0
            and numeric_values.notna().all()
            and numeric_values.reset_index(drop=True).equals(
                pd.Series(range(len(df)), dtype=numeric_values.dtype)
            )
        )

        if looks_like_index:
            recommendation = "Likely exported index — review for removal"
        else:
            recommendation = "Needs manual review — keep for now"

        review_results.append({
            "column": col,
            "missing_values": int(values.isna().sum()),
            "unique_values": int(values.nunique()),
            "sample_values": values.head(5).tolist(),
            "recommendation": recommendation
        })

    return pd.DataFrame(review_results)


In [ ]:
def profile_dataset(df):
    """Generate a basic profile for any tabular dataset."""

    profile = {
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": df.columns.tolist(),
        "data_types": {
            col: str(df[col].dtype)
            for col in df.columns
        },
        "missing_values": {
            col: int(df[col].isna().sum())
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "missing_percentage": {
            col: round(
                df[col].isna().mean() * 100, 2
            )
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "duplicate_rows": int(df.duplicated().sum()),
        "unique_values": {
            col: int(df[col].nunique())
            for col in df.columns
        }
    }

    return profile


In [ ]:
import pandas as pd

def run_data_quality_checks(df):
    """
    Identify common data-quality issues in any DataFrame.
    Does not modify or delete the original data.
    """

    results = []

    total_rows = len(df)

    for col in df.columns:
        series = df[col]
        missing_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))

        # 1. Check missing values and completely empty columns
        if total_rows > 0 and missing_count == total_rows:
            results.append({
                "column": col,
                "issue": "Completely empty column",
                "count": missing_count,
                "severity": "High",
                "recommendation":
                    "All values in this column are NaN/null. "
                    "Consider deleting this column."
           })

        elif missing_count > 0:
            results.append({
                "column": col,
                "issue": "Missing values",
                "count": missing_count,
                "severity": (
                    "High" if missing_count / total_rows >= 0.5
                    else "Medium"
                ),
                "recommendation":
                    "Review missing values before deciding how to handle them."
            })

         # 2. Check constant columns
        if total_rows > 0 and unique_count == 1 :
            results.append({
                "column": col,
                "issue": "Constant",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column provides useful information."
            })

        # 3. Potential numeric data stored as text
        if pd.api.types.is_object_dtype(series) or \
           pd.api.types.is_string_dtype(series):

            non_missing = series.dropna()

            if len(non_missing) > 0:
                converted = pd.to_numeric(
                    non_missing.astype(str).str.replace(",", "", regex=False),
                    errors="coerce"
                )

                numeric_ratio = converted.notna().mean()

                if numeric_ratio >= 0.8:
                    results.append({
                        "column": col,
                        "issue": "Possible numeric data stored as text",
                        "count": int(converted.notna().sum()),
                        "severity": "Medium",
                        "recommendation":
                            "Review the values and convert to numeric if appropriate."
                    })

        # 4. High-cardinality columns
        if total_rows > 0 and unique_count / total_rows >= 0.9:
            results.append({
                "column": col,
                "issue": "High cardinality",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column contains IDs or mostly unique values."
            })

    # 5. Duplicate rows (dataset-level issue)
    duplicate_count = int(df.duplicated().sum())

    # 6. Identical columns
    results.extend(find_identical_columns(df))

    if duplicate_count > 0:
        results.append({
            "column": "(entire dataset)",
            "issue": "Duplicate rows",
            "count": duplicate_count,
            "severity": "Medium",
            "recommendation":
                "Review duplicate records before removing them."
        })

    return pd.DataFrame(
        results,
        columns=[
            "column", "issue", "count",
            "severity", "recommendation"
        ]
    )


In [ ]:
from itertools import combinations

def find_identical_columns(df):
    """Find exact duplicate columns without changing the dataset."""

    findings = []
    candidate_groups = {}

    # 1. Create a fingerprint for each column
    for position in range(df.shape[1]):
        series = df.iloc[:, position]

        fingerprint = pd.util.hash_pandas_object(
            series, index=False
        ).values.tobytes()

        candidate_groups.setdefault(
            fingerprint, []
        ).append(position)

    # 2. Compare only columns with matching fingerprints
    for positions in candidate_groups.values():
        if len(positions) < 2:
            continue

        for i, j in combinations(positions, 2):
            first = df.iloc[:, i]
            second = df.iloc[:, j]

            # Verify actual equality; never rely on hashes alone
            if first.equals(second):
                findings.append({
                    "column": str(df.columns[i]),
                    "issue": "Identical columns",
                    "count": len(df),
                    "severity": "Medium",
                    "recommendation": (
                        f"Column at position {j} "
                        f"('{df.columns[j]}') is identical to "
                        f"column at position {i} "
                        f"('{df.columns[i]}'). Review before "
                        "removing either column."
                    )
                })

    return findings


In [ ]:
duplicate_column_findings = find_identical_columns(df)

print("Identical column pairs found:",
      len(duplicate_column_findings))

display(pd.DataFrame(duplicate_column_findings))

Identical column pairs found: 2


,column,issue,count,severity,recommendation
0,transmission,Identical columns,8369,Medium,Column at position 25 ('Transmission') is iden...
1,priceSaving,Identical columns,8369,Medium,Column at position 16 ('priceFixedText') is id...


In [ ]:
results=[]
for col in df.columns:

  #print(type(col))
  #print(col)
  '''series = df[col]
        missing_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))
        total_rows = len(df)
        print(series)
        print(f'missing value = {missing_count}')
        print(f'unique value = {unique_count}')
        print(f'total rows = {total_rows}')'''

df["owner"].nunique(dropna=True)


6

In [ ]:
quality_report = run_data_quality_checks(df)

display(quality_report)

,column,issue,count,severity,recommendation
0,it,Constant,1,Low,Check whether this column provides useful info...
1,bt,Missing values,4,Medium,Review missing values before deciding how to h...
2,km,Possible numeric data stored as text,8369,Medium,Review the values and convert to numeric if ap...
3,priceActual,Missing values,6699,High,Review missing values before deciding how to h...
4,priceSaving,Completely empty column,8369,High,All values in this column are NaN/null. Consid...
5,priceFixedText,Completely empty column,8369,High,All values in this column are NaN/null. Consid...
6,Registration Year,Missing values,52,Medium,Review missing values before deciding how to h...
7,Insurance Validity,Missing values,4,Medium,Review missing values before deciding how to h...
8,Seats,Missing values,6,Medium,Review missing values before deciding how to h...
9,Kms Driven,Missing values,2,Medium,Review missing values before deciding how to h...


In [ ]:
df.shape

(8369, 34)

In [ ]:
print("Total dataset columns:", df.shape[1])
print("Total quality findings:", len(quality_report))
print(
    "Unique columns with findings:",
    quality_report["column"].nunique()
)

Total dataset columns: 34
Total quality findings: 21
Unique columns with findings: 20


In [ ]:
columns_with_findings = set(quality_report["column"])

columns_without_findings = [
    col for col in df.columns
    if col not in columns_with_findings
]

print("Columns without detected findings:",
      len(columns_without_findings))

print(columns_without_findings)

Columns without detected findings: 14
['Unnamed: 0', 'ft', 'transmission', 'ownerNo', 'owner', 'oem', 'model', 'modelYear', 'centralVariantId', 'variantName', 'price', 'Fuel Type', 'Transmission', 'top_features']


In [ ]:
display(
    quality_report[
        ["column", "issue", "count", "severity", "recommendation"]
    ].sort_values(["column", "issue"]).reset_index(drop=True)
)

,column,issue,count,severity,recommendation
0,Engine,Missing values,4,Medium,Review missing values before deciding how to h...
1,Engine Displacement,Missing values,4,Medium,Review missing values before deciding how to h...
2,Insurance Validity,Missing values,4,Medium,Review missing values before deciding how to h...
3,Kms Driven,Missing values,2,Medium,Review missing values before deciding how to h...
4,Max Power,Missing values,60,Medium,Review missing values before deciding how to h...
5,Mileage,Missing values,287,Medium,Review missing values before deciding how to h...
6,Ownership,Missing values,32,Medium,Review missing values before deciding how to h...
7,RTO,Missing values,894,Medium,Review missing values before deciding how to h...
8,Registration Year,Missing values,52,Medium,Review missing values before deciding how to h...
9,Seats,Missing values,6,Medium,Review missing values before deciding how to h...


In [ ]:

columns_to_inspect = [
    "it",
    "km",
    "Wheel Size",
    "priceActual",
    "price",
    "Unnamed: 0",
    "Seats",
    "Seats.1",
    "transmission",
    "Transmission"
]

for col in columns_to_inspect:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print("Data type:", df[col].dtype)
        print("Missing values:", df[col].isna().sum())
        print("Unique values:", df[col].nunique())
        print("Sample values:")
        print(df[col].dropna().head(5).tolist())



--- it ---
Data type: int64
Missing values: 0
Unique values: 1
Sample values:
[0, 0, 0, 0, 0]

--- km ---
Data type: object
Missing values: 0
Unique values: 4090
Sample values:
['1,20,000', '32,706', '11,949', '17,794', '60,000']

--- Wheel Size ---
Data type: object
Missing values: 2983
Unique values: 17
Sample values:
['16', '14', '14', '16', '17']

--- priceActual ---
Data type: object
Missing values: 6699
Unique values: 721
Sample values:
['₹ 5.85 Lakh', '₹ 8.21 Lakh', '₹ 4.11 Lakh', '₹ 6.99 Lakh', '₹ 4.20 Lakh']

--- price ---
Data type: object
Missing values: 0
Unique values: 1400
Sample values:
['₹ 4 Lakh', '₹ 8.11 Lakh', '₹ 5.85 Lakh', '₹ 4.62 Lakh', '₹ 7.90 Lakh']

--- Unnamed: 0 ---
Data type: int64
Missing values: 0
Unique values: 1485
Sample values:
[0, 1, 2, 3, 4]

--- Seats ---
Data type: object
Missing values: 6
Unique values: 8
Sample values:
['5 Seats', '5 Seats', '5 Seats', '5 Seats', '5 Seats']

--- Seats.1 ---
Data type: float64
Missing values: 6
Unique values: 8
S

In [ ]:
column_pairs = [
    ("Seats", "Seats.1"),
    ("transmission", "Transmission")
]

for col1, col2 in column_pairs:
    if col1 in df.columns and col2 in df.columns:
        print(
            f"{col1} vs {col2}:",
            df[col1].equals(df[col2])
        )


Seats vs Seats.1: False
transmission vs Transmission: True


In [ ]:

# Check whether km values can be converted to numbers
km_clean = pd.to_numeric(
    df["km"].astype("string").str.replace(",", "", regex=False),
    errors="coerce"
)

print("km values that failed conversion:", km_clean.isna().sum())
print("km values that are negative:", (km_clean < 0).sum())

# Check Wheel Size conversion
wheel_clean = pd.to_numeric(df["Wheel Size"], errors="coerce")

invalid_wheel_values = (
    df["Wheel Size"].notna() & wheel_clean.isna()
)

print("Wheel Size values that failed conversion:",
      invalid_wheel_values.sum())

if invalid_wheel_values.any():
    print(df.loc[invalid_wheel_values, "Wheel Size"].unique())


km values that failed conversion: 0
km values that are negative: 0
Wheel Size values that failed conversion: 341
['R18' 'R16' 'R17' 'R15' 'R19' 'R14' 'R20']


In [ ]:

# Keep the original column unchanged
wheel_text = df["Wheel Size"].astype("string").str.strip()

# Extract the numeric part from values such as 16 and R18
wheel_number = pd.to_numeric(
    wheel_text.str.extract(r"^[Rr]?(\d+(?:\.\d+)?)$", expand=False),
    errors="coerce"
)

# Identify non-missing values that do not match the expected format
invalid_wheel = df["Wheel Size"].notna() & wheel_number.isna()

print("Original missing values:", df["Wheel Size"].isna().sum())
print("Values successfully interpreted:", int(wheel_number.notna().sum()))
print("Values needing manual review:", int(invalid_wheel.sum()))

if invalid_wheel.any():
    print("Values needing review:")
    print(df.loc[invalid_wheel, "Wheel Size"].value_counts())


Original missing values: 2983
Values successfully interpreted: 5386
Values needing manual review: 0


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

duplicate_rows = df[df.duplicated(keep=False)]

display(duplicate_rows.head(10))

Duplicate rows: 0


,Unnamed: 0,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Engine Displacement,Transmission,Year of Manufacture,top_features,Mileage,Engine,Max Power,Torque,Seats.1,Wheel Size


In [ ]:
import json
import pandas as pd

def process_dataset(file_path):
    file_path = Path(file_path)

    print(f"\nProcessing: {file_path.name}")

    # 1. Load dataset
    df = load_dataset(file_path)

    # 2. Profile dataset
    profile = profile_dataset(df)

    # 3. Review suspicious Unnamed columns
    unnamed_report = review_unnamed_columns(df)

    # 4. Create missing-value report
    missing_report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (df.isna().mean() * 100).round(2)
    })

    missing_report = missing_report[
        missing_report["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)

    # 5. Run data-quality checks
    quality_report = run_data_quality_checks(df)

    # 6. Save reports using the dataset's filename
    report_name = file_path.stem

    with open(REPORT_DIR / f"{report_name}_profile.json", "w") as f:
        json.dump(profile, f, indent=4, default=str)

    missing_report.to_csv(
        REPORT_DIR / f"{report_name}_missing_values.csv"
    )

    unnamed_report.to_csv(
        REPORT_DIR / f"{report_name}_column_review.csv",
        index=False
    )

    quality_report.to_csv(
        REPORT_DIR / f"{report_name}_data_quality_report.csv",
        index=False
    )


    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Reports saved to:", REPORT_DIR)

    return df, profile, missing_report, unnamed_report
